In [1]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

# Load your code.env file
env_file = r"C:\Users\Abhijeet\code.env"

with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

print(f" Loaded from: {env_file}")

ok_openai = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI : {' Loaded' if ok_openai else ' Not found'}")

 Loaded from: C:\Users\Abhijeet\code.env
OpenAI :  Loaded


In [3]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"],
               capture_output=True)

# Load from code.env
env_file = r"C:\Users\Abhijeet\code.env"
with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

# Anthropic not needed — set dummy so scraper skips quietly
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = "not-used"

ok = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI key : {' Ready' if ok else ' Not found — check code.env'}")
if ok:
    print(" Ready to run scraper")

OpenAI key :  Ready
 Ready to run scraper


In [4]:
import subprocess, sys
for lib in ["playwright", "beautifulsoup4", "nest_asyncio", "python-dotenv"]:
    subprocess.run([sys.executable, "-m", "pip", "install", lib, "-q"], capture_output=True)
subprocess.run("sudo playwright install-deps chromium 2>/dev/null || true", shell=True, capture_output=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"])
print("✓ All libraries installed")

✓ All libraries installed


In [6]:
import asyncio
import re
import sys
import json
import urllib.request
import os
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime


# ── SENTENCE-SAFE TRUNCATION ──────────────────────────────────────────────────

def clean_to_sentences(text: str, max_sentences: int = 2) -> str:
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    good = [
        s.strip() for s in sentences
        if len(s.strip().split()) >= 6 and re.search(r'[.!?]$', s.strip())
    ]
    return " ".join(good[:max_sentences])


# ── API KEY ───────────────────────────────────────────────────────────────────

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def _get_openai_key() -> str:
    return os.environ.get("OPENAI_API_KEY", "")

def _check_key(key: str, name: str) -> bool:
    if not key or key.startswith("YOUR_") or len(key) < 20:
        print(f"  ⚠ {name} not set — add it to your .env file")
        return False
    return True


# ── HELPERS ───────────────────────────────────────────────────────────────────

async def highlight_and_click(page, selector_or_locator, description="Action"):
    try:
        target = (
            page.locator(selector_or_locator).first
            if isinstance(selector_or_locator, str)
            else selector_or_locator
        )
        if await target.is_visible(timeout=2000):
            await target.evaluate(
                "el => { el.style.border = '4px solid #00FF00';"
                " el.style.backgroundColor = 'rgba(0,255,0,0.2)'; }"
            )
            await target.click()
            return True
    except Exception:
        pass
    return False


async def handle_cookies_automatically(page):
    for selector in [
        "text=Accept All", "text=Accept", "text=Accepter",
        "text=Tout accepter", "text=Accepter tout",
        "button:has-text('Accept')", "button:has-text('Accepter')",
        "button:has-text('OK')", "button:has-text('I Agree')",
        "button:has-text('J\\'accepte')", "button:has-text('Close')",
        "button:has-text('Got it')", "button:has-text('Agree')",
        "button:has-text('Continuer')", "button:has-text('Fermer')",
        "button:has-text('Refuser')", "button:has-text('Reject all')",
    ]:
        if await highlight_and_click(page, selector, "Cookie Banner"):
            break


async def smart_goto(page, url, timeout=90000):
    try:
        await page.goto(url, wait_until="networkidle", timeout=timeout)
        return True
    except Exception as e:
        print(f"  ⚠ networkidle failed ({e.__class__.__name__}), retrying…")
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout)
        await asyncio.sleep(6)
        try:
            await page.wait_for_function(
                "() => document.body && document.body.innerText.trim().length > 200",
                timeout=15000)
        except Exception:
            pass
        return True
    except Exception as e:
        try:
            await page.goto(url, wait_until="commit", timeout=timeout)
            await asyncio.sleep(8)
            return True
        except Exception:
            pass
        print(f"  ✗ Failed: {e}")
        return False


# ── KNOWN NAMES MAP — city locked, always wins over page title ────────────────

KNOWN_NAMES_PARIS = {
    # domain_fragment: (correct_name, correct_city)
    "alsacienne.org":                 ("École Alsacienne",                                    "Paris, France"),
    "janson-de-sailly.fr":            ("Lycée Janson de Sailly",                              "Paris, France"),
    "lycee-international-stgermain":  ("Lycée International de Saint-Germain-en-Laye",        "Saint-Germain-en-Laye, France"),
    "breteuil.fr":                    ("Château de Breteuil",                                  "Choisel, France"),
    "stanislas.fr":                   ("Collège-Lycée Stanislas",                              "Paris, France"),
    "psg.fr":                         ("PSG Academy",                                          "Paris, France"),
    "rolandgarros.com":               ("Roland-Garros",                                        "Paris, France"),
    "racingclubdefrance.fr":          ("Racing Club de France",                                "Paris, France"),
    "poloclubchantilly.com":          ("Chantilly Polo Club",                                  "Chantilly, France"),
    "axiom-academic.com":             ("Axiom Academic",                                       "Paris, France"),
    "cours-thales.fr":                ("Cours Thalès",                                         "Paris, France"),
    "icp.fr":                         ("Institut Catholique de Paris",                         "Paris, France"),
    "sciencespo.fr":                  ("Sciences Po Paris",                                    "Paris, France"),
    "hec.edu":                        ("HEC Paris",                                            "Jouy-en-Josas, France"),
    "operadeparis.fr":                ("Opéra national de Paris",                              "Paris, France"),
    "conservatoiredeparis.fr":        ("Conservatoire National Supérieur de Musique de Paris", "Paris, France"),
    "rye-yoga.fr":                    ("RYE — Réseau Yoga Éducation",                          "Paris, France"),
    "ecolejeanninemanuel.org":        ("École Jeannine Manuel",                                "Paris, France"),
    "saintemariedeneuilly.fr":        ("Sainte-Marie de Neuilly",                              "Neuilly-sur-Seine, France"),
}

# Hardcoded verified facts — never overwritten by scraping
# Founded sources: Wikipedia / official history pages
KNOWN_DATA_PARIS = {
    "alsacienne.org": {
        "Founded": "1874", "Ages": "3–18", "Students": "1800",
        "Type": "Independent Day School",
    },
    "janson-de-sailly.fr": {
        "Founded": "1884", "Ages": "11–18", "Students": "3500",
        "Type": "Lycée with Classes Préparatoires",
    },
    "lycee-international-stgermain": {
        "Founded": "1952", "Ages": "3–18", "Students": "4000",
        "Type": "International Lycée",
    },
    "breteuil.fr": {
        "Founded": "1712", "Ages": "All ages", "Students": "N/A",
        "Type": "Heritage & Cultural Site",
    },
    "stanislas.fr": {
        "Founded": "1804", "Ages": "3–18", "Students": "4000",
        "Type": "Catholic Independent School",
    },
    "psg.fr": {
        "Founded": "2005", "Ages": "4–17", "Students": "N/A",
        "Type": "Football Academy",
    },
    "rolandgarros.com": {
        "Founded": "1891", "Ages": "All ages", "Students": "N/A",
        "Type": "Grand Slam Tennis Tournament & Venue",
    },
    "racingclubdefrance.fr": {
        "Founded": "1882", "Ages": "All ages", "Students": "N/A",
        "Type": "Sports & Social Club",
    },
    "poloclubchantilly.com": {
        "Founded": "1898", "Ages": "All ages", "Students": "N/A",
        "Type": "Equestrian & Polo Club",
    },
    "axiom-academic.com": {
        "Founded": "2016", "Ages": "5–18", "Students": "N/A",
        "Type": "Academic Tutoring Service",
    },
    "cours-thales.fr": {
        "Founded": "2007", "Ages": "15–25", "Students": "2400",
        "Type": "Classes Préparatoires & Test Prep",
    },
    "icp.fr": {
        "Founded": "1875", "Ages": "18–30", "Students": "10000",
        "Type": "Catholic University",
    },
    "sciencespo.fr": {
        "Founded": "1872", "Ages": "18–30", "Students": "14000",
        "Type": "Grande École — Political Science",
    },
    "hec.edu": {
        "Founded": "1881", "Ages": "18–30", "Students": "4500",
        "Type": "Grande École — Business",
    },
    "operadeparis.fr": {
        "Founded": "1669", "Ages": "All ages", "Students": "N/A",
        "Type": "National Opera & Ballet Company",
    },
    "conservatoiredeparis.fr": {
        "Founded": "1795", "Ages": "18–30", "Students": "1400",
        "Type": "National Music & Dance Conservatory",
    },
    "rye-yoga.fr": {
        "Founded": "1978", "Ages": "All ages", "Students": "N/A",
        "Type": "Yoga Education Organisation",
    },
    "ecolejeanninemanuel.org": {
        "Founded": "1954", "Ages": "3–18", "Students": "2200",
        "Type": "Bilingual International School",
    },
    "saintemariedeneuilly.fr": {
        "Founded": "1846", "Ages": "3–18", "Students": "1200",
        "Type": "Catholic Independent School",
    },
}


# ── SHORT KEYWORDS MAP for WP blocks ──────────────────────────────────────────

PERF_SHORT = {
    "Coaching Credentials":   "Expert Staff",
    "Student Wellbeing":      "Student Care",
    "Academic Integration":   "Curriculum",
    "Competitive Pathway":    "Achievements",
    "Facilities & Resources": "Facilities",
    "Ongoing Accountability": "Progress",
}


# ── GARBAGE SITE DETECTION ────────────────────────────────────────────────────

GARBAGE_SIGNALS = [
    "hugedomains", "domain for sale", "buy this domain", "sedoparking",
    "this domain is for sale", "godaddy", "namecheap parking",
    "page not found", "account suspended", "coming soon",
    "under construction", "website coming soon",
]

def is_garbage_site(name: str, body_text: str) -> bool:
    combined = (name + " " + body_text[:500]).lower()
    return any(sig in combined for sig in GARBAGE_SIGNALS)


# ── JUNK FILTER ───────────────────────────────────────────────────────────────

JUNK_PATTERNS = [
    # UI / navigation
    r"the site you are about to visit", r"page you requested no longer exists",
    r"managed independently", r"beginning of dialog window", r"escape will cancel",
    r"enquire now$", r"apply now$", r"watch the video", r"learn more$",
    r"find out more$", r"read more$", r"click here",
    r"cookie", r"privacy policy", r"gdpr", r"mentions légales",
    r"droits réservés", r"tous droits", r"politique de confidentialité",
    # Cookie consent
    r"use the .accept all. button", r"use the .reject all. button",
    r"continue without accepting", r"consent to the use",
    r"gérer les cookies", r"gestion des cookies",
    r"identifiez-vous pour accéder", r"créez votre compte",
    r"votre adresse de messagerie est uniquement utilisée",
    r"lien de désabonnement intégré",
    # Form / UI
    r"please fill out the form", r"fill the registration form",
    r"merci d'indiquez vos informations", r"recevoir votre tarif personnalisé",
    r"nos conseillers sont disponibles de 9h",
    r"un conseiller pédagogique vous contacte",
    # Sport news / scores (for Roland-Garros, Racing Club etc.)
    r"régional d[12] j\d+", r"s'incline \d+ (à|a) \d+",
    r"face à .{3,40} \d+ (à|a) \d+",
    r"victoire de l'entente", r"magnifique victoire",
    r"injured to his wrist", r"forced to withdraw",
    r"two-time defending champion",
    r"pulling out of the barcelona tournament",
    r"after the results of the tests carried out today",
    r"most prudent thing to do is to be cautious",
    r"this is a difficult time for me",
    # Heritage / non-school
    r"ouverture du parc", r"dernier billet vendu",
    r"fermeture du parc", r"par téléphone.*lundi",
    r"tarif plein", r"tarif réduit", r"moins de 5 ans",
    r"tad\.iledefrance",
    # Misc junk
    r"cette expression fut utilisée par jules ferry",
    r"mésentente au sein de son couple",
    r"don de l'intégralité de ses biens",
    r"single.elimination system",
    r"mailing list.*keep you up to date",
    r"to progress your application we require",
    r"our system is limited to uploading",
    r"mai 1968 se passe en interne",
    r"les années 1970 voient le lycée international",
    r"jardin des chefs.*michelin",
    r"1,200 m² terrace restaurant",
    r"leading chefs and pastry chefs",
    r"entry lists for all roland",
    r"article will be regularly updated",
    r"compagnie du renault",
    r"bnp paribas",
    # ICP / conservatoire junk
    r"bernard o'mahony.*armées",
    r"école des commissaires de la marine",
    r"work package 1.*coordinated by",
    r"norwegian academy of music",
    r"in.tune brings together 8 institutions",
    # École Jeannine Manuel cookie
    r"accept all.*button to consent",
    r"reject all.*button or close",
    # Axiom outcomes junk
    r"nos conseillers sont disponibles",
    r"toute inscription est conditionnée",
    r"ent \| portail cdi",
]

def is_junk(text: str) -> bool:
    if not text:
        return True
    tl = text.lower().strip()
    return any(re.search(p, tl) for p in JUNK_PATTERNS)


# ── IMAGE HARVESTING ──────────────────────────────────────────────────────────

SKIP_IMG = [
    "logo", "icon", "svg", "1x1", "pixel", "spinner", "placeholder",
    "blank", "transparent", "avatar", "flag", "arrow", "bullet",
    "check", "star", "rating", "rs/facebook", "rs/linkedin", "rs/youtube",
    "loupe", "panier", "menu", "profil", "derouler",
    "default_image", "black-beacon", "weather/128",
    "renault.jpg", "bnpparibas.png", "engie.png",  # RG sponsor logos
    "le-monde-magazine", "le-figaro", "les-echos",  # Press logo images
    "retour.png", "catalogue.png",                  # UI buttons
    "catalogue-480", "pdf-300",
]
EXTS_IMG = [".jpg", ".jpeg", ".png", ".webp"]

async def harvest_images(pg, base_url: str, image_set: set):
    try:
        await pg.evaluate("""async () => {
            await new Promise(resolve => {
                let total = 0;
                const dist = 400;
                const timer = setInterval(() => {
                    window.scrollBy(0, dist);
                    total += dist;
                    if (total >= document.body.scrollHeight) {
                        clearInterval(timer);
                        window.scrollTo(0, 0);
                        resolve();
                    }
                }, 80);
            });
        }""")
        await asyncio.sleep(1)
    except Exception:
        pass

    try:
        for img in await pg.query_selector_all("img"):
            for attr in ["src", "data-src", "data-lazy-src", "data-original",
                         "data-image", "data-bg", "data-lazy", "data-srcset"]:
                c = await img.get_attribute(attr)
                if c:
                    src = c.strip().split(" ")[0].split(",")[0].strip()
                    full = urljoin(base_url, src)
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)
                            and len(full) > 25):
                        image_set.add(full)
            srcset = await img.get_attribute("srcset")
            if srcset:
                parts = [p.strip().split(" ")[0] for p in srcset.split(",") if p.strip()]
                if parts:
                    full = urljoin(base_url, parts[-1])
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)):
                        image_set.add(full)

        for sel in ['meta[property="og:image"]', 'meta[name="twitter:image"]',
                    'meta[property="og:image:url"]']:
            try:
                for og in await pg.evaluate(
                    f"() => Array.from(document.querySelectorAll('{sel}')).map(m=>m.content)"
                ):
                    if og and og.strip() and not any(s in og.lower() for s in SKIP_IMG):
                        image_set.add(og.strip())
            except Exception:
                pass

        bg_urls = await pg.evaluate("""() => {
            const hits = [];
            const sels = [
                '[style*="background-image"]','[class*="hero"]','[class*="banner"]',
                '[class*="slider"]','[class*="slide"]','[class*="bg-image"]',
                '[class*="cover"]','section','header'
            ];
            sels.forEach(sel => {
                try {
                    document.querySelectorAll(sel).forEach(el => {
                        const s = el.style.backgroundImage ||
                                  getComputedStyle(el).backgroundImage || '';
                        const m = s.match(/url\\(["']?([^"')]+)["']?\\)/);
                        if (m && m[1]) hits.push(m[1]);
                    });
                } catch(e) {}
            });
            return [...new Set(hits)];
        }""")
        for bg in bg_urls:
            if bg:
                full = urljoin(base_url, bg.strip())
                if (any(e in full.lower() for e in EXTS_IMG)
                        and not any(s in full.lower() for s in SKIP_IMG)):
                    image_set.add(full)
    except Exception:
        pass


# ── QUOTE SCRAPER ─────────────────────────────────────────────────────────────

def _word_count(text):
    return len(text.split())

def scrape_quote_from_soup(soup):
    SKIP_PHRASES = [
        "cookie", "menu", "nav", "©", "privacy", "terms", "mentions légales",
        "subscribe", "sign up", "click", "read more", "javascript",
        "watch the video", "watch video", "listen to", "testimonial",
        "share", "follow us", "contact us", "apply now", "find out more",
        "learn more", "get in touch", "download", "register", "book a",
        "visit us", "dialog window", "escape will cancel",
        "site you are about to visit", "no longer exists",
        "identifiez-vous", "créez votre compte",
        "accept all", "reject all", "consent",
        "roland-garros 2026", "roland‑garros 2026",
        "entry lists for all",
        "nos conseillers sont disponibles",
        "votre adresse de messagerie",
    ]
    selectors = [
        "blockquote", "q",
        "[class*='quote']", "[class*='pullquote']", "[class*='callout']",
        "[class*='hero'] p", "[class*='banner'] p",
        "[class*='mission'] p", "[class*='vision'] p",
        "figcaption",
    ]
    candidates = []
    for sel in selectors:
        for el in soup.select(sel):
            text = re.sub(r'\s+', ' ', el.get_text()).strip()
            text = text.strip('\u201c\u201d\u2018\u2019"\'«»')
            wc = _word_count(text)
            if wc < 8 or wc > 35:
                continue
            if any(x in text.lower() for x in SKIP_PHRASES):
                continue
            if is_junk(text):
                continue
            if not re.search(r'[.!?]$', text):
                continue
            candidates.append(text)
    if not candidates:
        return ""
    candidates.sort(key=lambda t: _word_count(t))
    return candidates[0]


# ── AI QUOTE ──────────────────────────────────────────────────────────────────

def generate_ai_quote(school_name, school_type, city, about_text):
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return f"Excellence, integrity, and curiosity — the pillars of {school_name}."
    prompt = (
        f"Write ONE short inspirational quote (1 sentence, 10–20 words) "
        f"that reflects the philosophy of {school_name}, a {school_type} in {city}. "
        f"Context: {about_text[:300]}. "
        f"Sound like the organisation's mission statement. "
        f"Return ONLY the quote text — no quotation marks, no attribution, no explanation."
    )
    payload = json.dumps({
        "model": "gpt-4o", "max_tokens": 60, "temperature": 0.7,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            data  = json.loads(resp.read())
            quote = data["choices"][0]["message"]["content"].strip().strip('"\'«»')
            words = quote.split()
            if len(words) > 25:
                quote = " ".join(words[:25])
            print(f"  ✓ AI quote generated for {school_name}")
            return quote
    except Exception as e:
        print(f"  ⚠ AI quote failed: {e}")
    return f"Excellence, integrity, and curiosity — the pillars of {school_name}."


# ── OPENAI FORCE-FILL ALL PERFORMANCE FIELDS ──────────────────────────────────

def openai_force_fill_performance(results: dict) -> dict:
    """
    Force-fills ALL 6 performance fields.
    Any field blank, N/A, too short (<12 words), or containing junk
    gets a unique, substantive 2-sentence AI response.
    NEVER overwrites a genuinely good scraped value.
    """
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    name        = results.get("Name", "this organisation")
    entity_type = results.get("Type", "Independent School")
    city        = results.get("City", "Paris, France")
    about       = results.get("About", "")
    philosophy  = results.get("Philosophy", "")

    ctx_parts = [f"Name: {name}", f"Type: {entity_type}", f"City: {city}"]
    if about:
        ctx_parts.append(f"About: {about}")
    if philosophy:
        ctx_parts.append(f"Philosophy: {philosophy}")
    for k in ["Founded", "Ages", "Students", "Fees", "Outcomes", "Admissions"]:
        v = results.get(k, "")
        if v and v not in ("N/A", "See website", ""):
            ctx_parts.append(f"{k}: {v}")
    context = "\n".join(ctx_parts)

    PERF_PROMPTS = {
        "Coaching Credentials": (
            f"2 sentences (≤35 words total) about the qualifications, expertise, or professional "
            f"background of staff/coaches/teachers at {name}, a {entity_type} in {city}. "
            f"Mention certifications, experience level, or teaching specialisms. Be specific."
        ),
        "Student Wellbeing": (
            f"2 sentences (≤35 words total) about how {name} supports the mental, emotional, "
            f"or physical wellbeing of students/members. Mention counselling, advisory, "
            f"pastoral care, or community support. Be specific."
        ),
        "Academic Integration": (
            f"2 sentences (≤35 words total) about the curriculum, programmes, or learning approach "
            f"at {name}. Mention specific courses, qualifications, or teaching methods. Be specific."
        ),
        "Competitive Pathway": (
            f"2 sentences (≤35 words total) about the results, achievements, university "
            f"destinations, competition outcomes, or career pathways from {name}. "
            f"Mention tangible outcomes or success metrics. Be specific."
        ),
        "Facilities & Resources": (
            f"2 sentences (≤35 words total) about the physical facilities, campus spaces, "
            f"or resources at {name}. "
            f"Mention specific amenities — studios, labs, theatres, pools, courts, fields. Be specific."
        ),
        "Ongoing Accountability": (
            f"2 sentences (≤35 words total) about how {name} tracks student/member progress, "
            f"provides feedback, or monitors quality. "
            f"Mention progress reports, parent communication, or assessment systems. Be specific."
        ),
    }

    # Extended junk list for Paris-specific scraping noise
    PERF_JUNK_PHRASES = [
        "cookie", "privacy policy", "gdpr", "javascript",
        "créez votre compte", "identifiez-vous",
        "nos conseillers sont disponibles", "recevoir votre tarif",
        "entry lists for all roland", "article will be regularly updated",
        "injured to his wrist", "forced to withdraw",
        "le jardin des chefs", "michelin-starred",
        "1,200 m² terrace restaurant", "leading chefs",
        "régional d", "s'incline", "face à conflans",
        "victoire de l'entente", "magnifique victoire",
        "par téléphone", "ouverture du parc", "dernier billet",
        "cette expression fut utilisée par jules ferry",
        "mésentente au sein de son couple",
        "in.tune brings together", "norwegian academy",
        "work package 1", "coordinated by the norwegian",
        "bernard o'mahony", "école des commissaires",
        "accept all.*button to consent",
        "after joining the french resistance",  # Jeannine Manuel founder story ≠ coaching
        "toute inscription est conditionnée",
        "ent | portail cdi",
        "pour célébrer les 150 ans de l'icp",
        "mai 1968 se passe en interne",
        "hec has set up the first green zone",
        "building an ecologically",
        "at the very heart of hec paris",
        "sustainability & organizations",
    ]

    def needs_fill(val: str) -> bool:
        if not val or val.strip() in ("N/A", "", "N/a", "See website"):
            return True
        clean = re.sub(r'\s+', ' ', val).strip()
        if len(clean.split()) < 12:
            return True
        cl = clean.lower()
        return any(p in cl for p in PERF_JUNK_PHRASES) or is_junk(clean)

    missing = {
        metric: prompt
        for metric, prompt in PERF_PROMPTS.items()
        if needs_fill(results.get("Performance", {}).get(metric, ""))
    }

    if not missing:
        print("  ✓ All performance fields complete — no AI fill needed.")
        return results

    print(f"  → AI force-filling {len(missing)} performance field(s): {', '.join(missing.keys())}")

    field_lines = "\n".join(f'  "{k}": {v}' for k, v in missing.items())
    prompt = f"""You are an expert on Parisian educational institutions, arts organisations, sports clubs, and cultural venues.

KNOWN DATA about this institution:
{context}

Generate UNIQUE, FACTUAL, SPECIFIC content for ONLY these fields.
Each field must:
- Be exactly 2 sentences, ≤35 words total
- Describe a DIFFERENT aspect from the other fields
- Sound professional and informative (not generic)
- Use your knowledge of {name} in {city} to be accurate
- Never repeat facts from another field

Reply ONLY with valid JSON, keys exactly as shown below. No markdown, no extra keys:

{field_lines}

Return format: {{"field name": "2 sentence content here.", ...}}"""

    payload = json.dumps({
        "model": "gpt-4o",
        "max_tokens": 1200,
        "temperature": 0.4,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=40) as resp:
            data   = json.loads(resp.read())
            raw    = data["choices"][0]["message"]["content"].strip()
            raw    = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw    = re.sub(r"\s*```$",           "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            seen_vals   = set()
            for metric, val in filled.items():
                if not val or val.strip() in ("N/A", ""):
                    continue
                key_short = re.sub(r'\s+', ' ', val.strip().lower())[:100]
                if key_short in seen_vals:
                    continue
                seen_vals.add(key_short)
                if metric in PERF_PROMPTS and needs_fill(
                    results["Performance"].get(metric, "")
                ):
                    results["Performance"][metric] = str(val).strip()
                    filled_keys.append(metric)

            if filled_keys:
                print(f"  ✓ AI filled performance: {', '.join(filled_keys)}")
            else:
                print("  ✓ AI performance: nothing new added.")

    except json.JSONDecodeError as e:
        print(f"  ⚠ AI performance JSON error: {e}")
    except Exception as e:
        print(f"  ⚠ AI performance fill failed: {e}")

    return results


# ── OPENAI MAIN FIELDS GAP-FILL ───────────────────────────────────────────────

def openai_fill_missing_fields(results: dict) -> dict:
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    entity_type = results.get("Type", "Independent School")
    is_school   = any(w in entity_type.lower()
                      for w in ["school", "academy", "college", "lycée", "lycee",
                                "conservatory", "conservatoire", "université",
                                "university", "grande école"])
    name = results.get("Name", "this organisation")

    FILL_FIELDS = {
        "Founded":    "Year this organisation was founded (just the 4-digit year). If unknown reply N/A.",
        "Ages":       ("Student age range (e.g. '6–18'). If unknown reply N/A." if is_school
                       else "Age range of members/visitors (e.g. 'All ages', 'Adults 18+', '5–80')."),
        "Students":   ("Approximate enrolled students. If unknown reply N/A." if is_school
                       else "Approximate annual members/visitors. If unknown reply N/A."),
        "Ratio":      "Staff-to-student or instructor-to-client ratio (e.g. '1:8'). If unknown reply N/A.",
        "Fees":       "Annual tuition or membership in EUR. French private school range typically €5,000–€25,000/year. If unknown reply 'See website'.",
        "About":      f"2 sentences (30–45 words) describing {name}'s core identity, mission, and what makes it distinctive.",
        "Philosophy": f"2 sentences (30–45 words) on {name}'s specific teaching, coaching, or organisational approach.",
        "Outcomes":   f"2 sentences (30–45 words) on concrete outcomes from {name} — university placements, awards, results.",
        "Admissions": f"2 sentences (30–45 words) on how to apply or enrol at {name} — process, timeline, requirements.",
        "Tagline":    f"One compelling tagline (≤20 words) for {name}.",
    }

    ctx_lines = []
    for k in ["Name", "Type", "City", "URL", "Tagline", "About", "Philosophy",
              "Outcomes", "Admissions", "Founded", "Ages", "Students", "Ratio", "Fees"]:
        v = results.get(k, "")
        if v:
            ctx_lines.append(f"{k}: {v}")
    context = "\n".join(ctx_lines) or f"URL: {results.get('URL', '')}"

    def _needs(val):
        return not val or str(val).strip() in ("N/A", "See website", "Rolling admissions",
                                                "Year-round", "", "N/a")

    def _needs_para(field, val):
        if _needs(val):
            return True
        if field in ("About", "Philosophy", "Outcomes", "Admissions"):
            if len(str(val).split()) < 20:
                return True
            # Check if the scraped content is actually junk
            return is_junk(str(val))
        return False

    missing = {}
    for f, p in FILL_FIELDS.items():
        v = results.get(f, "")
        if f in ("About", "Philosophy", "Outcomes", "Admissions"):
            if _needs_para(f, v):
                missing[f] = p
        elif _needs(v):
            missing[f] = p

    if not missing:
        return results

    print(f"  → OpenAI filling {len(missing)} main field(s): {', '.join(missing.keys())}")

    field_instructions = "\n".join(f'  "{k}": {v}' for k, v in missing.items())
    prompt = f"""You are a Paris and French education, arts, and culture expert.
Use the known data AND your knowledge to fill missing fields accurately for {name}.

KNOWN DATA:
{context}

Fill ONLY these fields (reply with a JSON object, keys exactly as shown):
{field_instructions}

Rules:
- Use factual knowledge of this specific Paris / French institution.
- Fees: use € and per year. French private school range typically €5,000–€25,000/year.
- Ages: always provide a range, even for non-schools (e.g. 'All ages', 'Adults 18+', '6–18').
- Paragraph fields (About, Philosophy, Outcomes, Admissions): 2 full sentences, 30–45 words total. Never write fewer than 25 words.
- Each paragraph must be specific and substantive — not generic filler.
- Do NOT add commentary, markdown fences, or extra keys.
- Reply with valid JSON only."""

    payload = json.dumps({
        "model": "gpt-4o", "max_tokens": 1800, "temperature": 0.3,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {key}"}, method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data   = json.loads(resp.read())
            raw    = data["choices"][0]["message"]["content"].strip()
            raw    = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw    = re.sub(r"\s*```$",           "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            for fkey, val in filled.items():
                if not val or str(val).strip() in ("N/A", "See website", ""):
                    continue
                if fkey in FILL_FIELDS:
                    v = results.get(fkey, "")
                    if fkey in ("About", "Philosophy", "Outcomes", "Admissions"):
                        if _needs_para(fkey, v):
                            results[fkey] = str(val).strip()
                            filled_keys.append(fkey)
                    elif _needs(v):
                        results[fkey] = str(val).strip()
                        filled_keys.append(fkey)

            if filled_keys:
                print(f"  ✓ OpenAI filled: {', '.join(filled_keys)}")
            else:
                print("  ✓ OpenAI: nothing new to fill.")
    except Exception as e:
        print(f"  ⚠ OpenAI main fill failed: {e}")

    return results


# ── WP BLOCKS GENERATOR (6 images + PERF_SHORT keywords) ─────────────────────

def generate_wp_blocks(r):
    school_name  = r.get("Name", "")
    description  = r.get("Tagline", "")
    age          = r.get("Ages", "")
    students     = r.get("Students", "")
    ratio        = r.get("Ratio", "")
    founded      = r.get("Founded", "")
    school_type  = r.get("Type", "")
    city         = r.get("City", "")
    annual_fee   = r.get("Fees", "")
    about        = r.get("About", "")
    philosophy   = r.get("Philosophy", "")
    outcomes     = r.get("Outcomes", "")
    quote        = r.get("Quote", "")
    website_url  = r.get("URL", "")
    admissions   = r.get("Admissions", "")
    images       = r.get("Images", [])
    perf         = r.get("Performance", {})
    enquiry_open = r.get("EnquiryOpen", "Year-round")
    app_deadline = r.get("AppDeadline", "Rolling admissions")

    website_display = website_url.replace("https://", "").replace("http://", "").rstrip("/")
    city_slug  = city.lower().replace(", ", "-").replace(" ", "-").replace(",", "")
    city_short = city.split(",")[0].strip()

    # Always 6 image slots — pad with img0 if needed
    imgs = [img for img in images if img]
    while len(imgs) < 6:
        imgs.append(imgs[0] if imgs else "")
    img0, img1, img2, img3, img4, img5 = imgs[0], imgs[1], imgs[2], imgs[3], imgs[4], imgs[5]

    coaching       = perf.get("Coaching Credentials", "")
    wellbeing      = perf.get("Student Wellbeing", "")
    academic       = perf.get("Academic Integration", "")
    competitive    = perf.get("Competitive Pathway", "")
    facilities     = perf.get("Facilities & Resources", "")
    accountability = perf.get("Ongoing Accountability", "")

    kw_coaching       = PERF_SHORT["Coaching Credentials"]
    kw_wellbeing      = PERF_SHORT["Student Wellbeing"]
    kw_academic       = PERF_SHORT["Academic Integration"]
    kw_competitive    = PERF_SHORT["Competitive Pathway"]
    kw_facilities     = PERF_SHORT["Facilities & Resources"]
    kw_accountability = PERF_SHORT["Ongoing Accountability"]

    wp = f"""<!-- wp:image {{"id":440,"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img0}" alt="{school_name}" class="wp-image-440"/></figure>
<!-- /wp:image -->

<!-- wp:paragraph -->
<p><a href="elite-home.html">Elite</a> &#8250; <a href="elite-city-{city_slug}.html">{city_short}</a> &#8250; <a href="elite-program-academic.html">Elite Academic Programs</a></p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Kidrovia Elite — Listed</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Elite Academic Programs</h5>
<!-- /wp:heading --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":1}} -->
<h1 class="wp-block-heading"><em>{school_name}</em></h1>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{description}</p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{age}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ages</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{students}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Students</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{ratio}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ratio</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph --><p>{founded}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Founded</h5><!-- /wp:heading --></div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column {{"width":"66.66%"}} -->
<div class="wp-block-column" style="flex-basis:66.66%">
<!-- wp:heading --><h2 class="wp-block-heading">About <em>{school_name}</em></h2><!-- /wp:heading -->
<!-- wp:paragraph --><p>{about}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column {{"width":"33.33%"}} -->
<div class="wp-block-column" style="flex-basis:33.33%">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Institution Details</h5><!-- /wp:heading -->
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Type</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ages</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Students</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Ratio</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Founded</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">City</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Annual Fee</h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{school_type}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{age}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{students}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{ratio}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{founded}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{city}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{annual_fee}</h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Contact &amp; Enquiry</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading"><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display} &#8594;</a></h5><!-- /wp:heading -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">Quote</h5>
<!-- /wp:heading -->
<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->
<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">— {school_name}</h5>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Philosophy</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">How they <em>teach</em></h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{philosophy}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Outcomes</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Where students <em>go</em></h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{outcomes}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img1}" alt="{school_name} campus"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img2}" alt="{school_name} students"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img3}" alt="{school_name} activities"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img4}" alt="{school_name} facilities"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:image {{"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img5}" alt="{school_name} community"/></figure>
<!-- /wp:image -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":4}} -->
<h4 class="wp-block-heading">Our Assessment</h4>
<!-- /wp:heading -->
<!-- wp:heading -->
<h2 class="wp-block-heading"><strong>How {school_name} performs</strong></h2>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_coaching}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Coaching Credentials</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{coaching}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_wellbeing}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Student Wellbeing</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{wellbeing}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_academic}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Academic Integration</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{academic}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_competitive}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Competitive Pathway</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{competitive}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_facilities}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Facilities &amp; Resources</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{facilities}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">{kw_accountability}</h5><!-- /wp:heading -->
<!-- wp:heading {{"level":3}} --><h3 class="wp-block-heading">Ongoing Accountability</h3><!-- /wp:heading -->
<!-- wp:paragraph --><p>{accountability}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns">
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading --><h2 class="wp-block-heading">Admissions &amp;<br><em>How to Apply</em></h2><!-- /wp:heading -->
<!-- wp:paragraph --><p>{admissions}</p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
<!-- wp:column -->
<div class="wp-block-column">
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Enquiries Open</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{enquiry_open}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Application Deadline</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{app_deadline}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Annual Fee</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{annual_fee}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Location</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p>{city}</p><!-- /wp:paragraph -->
<!-- wp:heading {{"level":5}} --><h5 class="wp-block-heading">Website</h5><!-- /wp:heading -->
<!-- wp:paragraph --><p><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display}</a></p><!-- /wp:paragraph -->
</div>
<!-- /wp:column -->
</div>
<!-- /wp:columns -->

<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->
<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">— {school_name}</h5>
<!-- /wp:heading -->
"""
    return wp


# ── MAIN SCRAPER ──────────────────────────────────────────────────────────────

async def extract_school_data(url):
    async with async_playwright() as p:
        import platform
        headless = os.environ.get(
            "HEADLESS",
            "true" if platform.system() == "Linux" else "false"
        ).lower() == "true"
        browser = await p.chromium.launch(headless=headless, slow_mo=0 if headless else 200)
        context = await browser.new_context(viewport={"width": 1280, "height": 800})
        page    = await context.new_page()

        results = {
            "Name": "", "Tagline": "", "Founded": "", "City": "",
            "Ages": "", "Students": "", "Ratio": "", "Fees": "",
            "Type": "Independent School",
            "About": "", "Philosophy": "", "Outcomes": "",
            "Admissions": "", "Quote": "",
            "EnquiryOpen": "Year-round",
            "AppDeadline": "Rolling admissions",
            "Performance": {}, "URL": url, "Images": [],
        }

        global_text_pool = []

        # ── STEP 0: Lock name, city, and known data from domain map ──────────
        domain = url.split("/")[2].replace("www.", "")
        known_entry = next(
            ((k, v) for k, v in KNOWN_NAMES_PARIS.items() if k in domain or k in url),
            None
        )
        known_domain_key = None
        if known_entry:
            k_fragment, (k_name, k_city) = known_entry
            results["Name"] = k_name
            results["City"] = k_city   # CITY LOCK — never overridden by body text
            # Find matching KNOWN_DATA key
            known_domain_key = next(
                (dk for dk in KNOWN_DATA_PARIS if dk in domain or dk in url), None
            )
            if known_domain_key:
                for field, val in KNOWN_DATA_PARIS[known_domain_key].items():
                    results[field] = val
            print(f"  ✓ Known institution: {k_name} | City locked: {k_city}")

        FALLBACKS = {
            "Founded": "", "Ages": "", "Students": "", "Ratio": "N/A", "Fees": "See website",
        }

        # Fee patterns — EUR / € / French formats
        fee_patterns = [
            r'(\d[\d\s]+)\s*€\s*(?:par\s+(?:an|mois|trimestre)|/\s*(?:an|année))?',
            r'€\s*([\d\s,]+)(?:par\s+(?:an|mois|trimestre))?',
            r'EUR\s*([\d,]+)',
            r'scolarit[eé]\s*(?:annuelle)?\s*[:\-]?\s*([\d\s,]+)\s*€?',
            r'frais\s*(?:de\s*scolarit[eé])?\s*[:\-]?\s*([\d\s,]+)\s*€?',
            r'tarif[s]?\s*[:\-]?\s*([\d\s,]+)\s*€?',
            r'fee[s]?\s*(?:of|is|are|:)?\s*€\s*([\d,]+)',
            r'tuition\s*(?:fee)?s?\s*(?:of|is|:)?\s*€\s*([\d,]+)',
        ]

        ratio_patterns = [
            r'(\d+\s*:\s*\d+)\s*(?:student[- ]to[- ](?:faculty|teacher)|teacher[- ]to[- ]student|ratio)',
            r'ratio\s*(?:of\s*|élèves[/ ])?(\d+\s*:\s*\d+)',
            r'(\d+\s*:\s*\d+)\s*ratio',
            r'1\s*(?:teacher|professeur|enseignant)\s+(?:to|pour|per|:)\s+(\d+)\s*(?:students?|élèves?)',
            r'(\d+)\s*(?:students?|élèves?)\s+(?:per|pour)\s+(?:1\s+)?(?:teacher|professeur)',
            r'(\d+)\s*élèves?\s+par\s+(?:classe|groupe|enseignant)',
        ]

        age_patterns = [
            r'students?\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'ages?\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'from\s+age\s+(\d+)\s+to\s+(\d+)',
            r'de\s+(\d+)\s+[àa]\s+(\d+)\s+ans',
            r'(\d+)\s+[àa]\s+(\d+)\s+ans',
            r'(\d+)\s*[\u2013\-]\s*(\d+)\s*year[s\-]\s*old',
            r'grade[s]?\s+(\d+)\s*(?:through|to|-)\s*(\d+)',
            r'(?:from|year)\s+(\d+)\s+to\s+(?:year\s+)?(\d+)',
        ]

        student_patterns = [
            r'(\d[\d,\s]+)\s*(?:\+)?\s*(?:pupils?|students?|boys|girls)',
            r'(\d[\d,\s]+)\s*(?:\+)?\s*(?:élèves?|apprenants?|lycéens?|étudiants?)',
            r'(?:approximately|around|about|environ|plus de|près de)\s+([\d,\s]+)\s*(?:pupils?|students?|élèves?)',
            r'accueille\s+(?:environ\s+)?([\d,\s]+)\s*(?:élèves?|étudiants?)',
            r'community\s+of\s+(?:over\s+)?([\d,]+)\s*(?:students?|élèves?)',
            r'([\d,]+)\s*(?:\+)?\s*(?:student|pupil|learner|élève)',
        ]

        print(f"Connecting to: {url}...")
        loaded = await smart_goto(page, url)
        if not loaded:
            print(f"  ✗ Could not load {url}, skipping.")
            await browser.close()
            return None

        try:
            await handle_cookies_automatically(page)

            og_site   = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:site_name\"]'); return m ? m.content : ''; }")
            og_title  = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:title\"]'); return m ? m.content : ''; }")
            raw_title = await page.title()

            REJECT_NAMES   = {
                "home", "welcome", "index", "main", "", "welcome to",
                "accueil", "page not found", "404", "error", "redirecting",
                "luxury hotels and resorts", "visit paris",
            }
            REJECT_STARTS  = ("welcome to ", "home -", "home |", "visit ",
                              "accueil -", "accueil |")
            REJECT_PHRASES = [
                "roland-garros 2026", "roland‑garros 2026",
                "cours thalès transforme", "youth school programs",
                "professeurs particuliers d'excellence",
            ]

            def clean_name(candidate: str) -> str:
                # Split only on | or — (em-dash), NOT hyphens
                parts = re.split(r'\s*[|—]\s*', candidate)
                parts = [p.strip() for p in parts if p.strip()]
                GENERIC = {"home","welcome","index","main","accueil","page","site","portal",""}
                n = parts[0] if parts else candidate.strip()
                if n.lower() in GENERIC and len(parts) > 1:
                    n = parts[-1].strip()
                for prefix in ("welcome to ", "visit ", "accueil - "):
                    if n.lower().startswith(prefix):
                        n = n[len(prefix):].strip()
                return n

            # Only set name if not already locked from KNOWN_NAMES_PARIS
            if not results["Name"]:
                for candidate in [og_site, og_title, raw_title]:
                    name = clean_name(candidate)
                    if (name
                            and name.lower() not in REJECT_NAMES
                            and not any(name.lower().startswith(p) for p in REJECT_STARTS)
                            and not any(bad in name.lower() for bad in REJECT_PHRASES)
                            and len(name.split()) <= 10):
                        results["Name"] = name
                        break
                if not results["Name"]:
                    domain_part = url.split("/")[2].replace("www.", "")
                    results["Name"] = domain_part.split(".")[0].replace("-", " ").title()

            body_text = await page.inner_text("body")

            if len(body_text.strip()) < 300:
                print("  ⚠ Page body too short — scrolling to trigger JS render…")
                try:
                    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    await asyncio.sleep(3)
                    await page.evaluate("window.scrollTo(0, 0)")
                    await asyncio.sleep(2)
                    body_text = await page.inner_text("body")
                except Exception:
                    pass

            # Only set city from body if not already locked
            if not results["City"]:
                results["City"] = _detect_city_from_body(body_text, url)

            if is_garbage_site(results["Name"], body_text):
                print(f"  ✗ Garbage/parked site detected — skipping {url}")
                await browser.close()
                return None

            def extract(pattern):
                m = re.search(pattern, body_text, re.IGNORECASE)
                return m.group(1).strip() if m else ""

            current_year = datetime.now().year

            def extract_founded_from_text(text, existing_candidates=None):
                STRONG_KWS = [
                    "founded in", "established in", "was founded", "founding year",
                    "founded by", "opened in", "inaugurated in",
                    "fondé en", "fondée en", "créé en", "créée en",
                    "ouvert en", "ouverte en", "inauguré en", "inaugurée en",
                    "a ouvert ses portes", "a été fondé", "a été créé",
                ]
                WEAK_KWS  = ["founded", "established", "foundation",
                             "fondé", "créé", "depuis", "histoire", "history"]
                SKIP_WORDS = ["copyright", "©", "droits réservés", "class of",
                              "alumni", "promotion", "concours", "résultats du bac"]
                FOREIGN_CTX = ["in the uk", "in england", "parent school", "sister school"]
                cands = existing_candidates if existing_candidates is not None else []
                clean = re.sub(r'\s+', ' ', text.lower())
                clean = re.sub(r'(copyright|©|droits réservés).*?\d{4}', '', clean)
                for sent in re.split(r'[.!?\n]', clean):
                    if any(w in sent for w in SKIP_WORDS):
                        continue
                    has_strong = any(k in sent for k in STRONG_KWS)
                    has_weak   = any(k in sent for k in WEAK_KWS)
                    if not (has_strong or has_weak):
                        continue
                    is_foreign = any(ctx in sent for ctx in FOREIGN_CTX)
                    for y in re.findall(r'\b(1[6-9]\d{2}|20[012]\d)\b', sent):
                        year = int(y)
                        if not (1600 <= year <= current_year - 1):
                            continue
                        score = 3 if has_strong else 1
                        if year < 1850:
                            score -= 2
                        if is_foreign:
                            score -= 3
                        if year >= current_year - 2:
                            score -= 6
                        elif year >= current_year - 5 and not has_strong:
                            score -= 3
                        cands.append((score, year))
                return cands

            # Only scrape Founded if not locked
            if not results.get("Founded"):
                founded_candidates = extract_founded_from_text(body_text)
                if founded_candidates:
                    founded_candidates.sort(key=lambda x: (-x[0], x[1]))
                    results["Founded"] = str(founded_candidates[0][1])

            # Only scrape Ages if not locked
            if not results.get("Ages"):
                age_candidates = []
                for pat in age_patterns:
                    for m in re.finditer(pat, body_text, re.I):
                        try:
                            g = m.groups()
                            if len(g) >= 2 and g[0] and g[1]:
                                lo, hi = int(g[0]), int(g[1])
                                if lo < hi and lo >= 2 and hi <= 25:
                                    age_candidates.append((lo, hi))
                        except Exception:
                            pass
                if age_candidates:
                    best = max(age_candidates, key=lambda x: x[1] - x[0])
                    results["Ages"] = f"{best[0]}–{best[1]}"

                if not results["Ages"]:
                    AGE_MAP = {
                        "petite section": (2,3), "moyenne section": (3,4),
                        "grande section": (4,5), "maternelle": (3,6),
                        "cp": (6,7), "ce1": (7,8), "ce2": (8,9),
                        "cm1": (9,10), "cm2": (10,11), "primaire": (6,11),
                        "sixième": (11,12), "cinquième": (12,13),
                        "quatrième": (13,14), "troisième": (14,15), "collège": (11,15),
                        "seconde": (15,16), "première": (16,17),
                        "terminale": (17,18), "lycée": (15,18),
                        "nursery": (2,4), "pre-k": (4,5), "kindergarten": (4,6),
                        "year 1": (5,6), "year 7": (11,12),
                        "year 13": (17,18), "sixth form": (16,18),
                        "a-level": (16,18),
                    }
                    tl = body_text.lower()
                    fy = [v for k, v in AGE_MAP.items() if k in tl]
                    results["Ages"] = f"{min(x[0] for x in fy)}–{max(x[1] for x in fy)}" if fy else ""

            # Only scrape Students if not locked
            if not results.get("Students"):
                for pat in student_patterns:
                    m = re.search(pat, body_text, re.I)
                    if m:
                        num_str = re.sub(r'[\s,]', '', m.group(1))
                        try:
                            num = int(num_str)
                            if 50 <= num <= 50000 and not (2020 <= num <= 2030):
                                results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                                break
                        except ValueError:
                            pass

            if not results.get("Ratio"):
                for pat in ratio_patterns:
                    m = re.search(pat, body_text, re.I)
                    if m:
                        results["Ratio"] = m.group(1).replace(" ", "")
                        break

            # Fee extraction
            if not results.get("Fees"):
                for pat in fee_patterns:
                    m = re.search(pat, body_text, re.I)
                    if m:
                        fee_str = m.group(0).strip()
                        nums = re.findall(r'\d+', fee_str.replace(' ','').replace('\xa0',''))
                        if nums:
                            n = int(nums[0])
                            if 100 <= n <= 100000 and not (2000 <= n <= 2030):
                                results["Fees"] = fee_str
                                break

            # Type detection — only if not locked from KNOWN_DATA_PARIS
            if not results.get("Type") or results["Type"] == "Independent School":
                tl    = body_text.lower()
                url_l = url.lower()
                if "soutien scolaire" in tl or "cours particuliers" in tl:
                    results["Type"] = "Academic Tutoring Service"
                elif "ib world" in tl or "international baccalaureate" in tl or "baccalauréat international" in tl:
                    results["Type"] = "IB World School"
                elif "classe préparatoire" in tl or "classes prépa" in tl:
                    results["Type"] = "Lycée with Classes Préparatoires"
                elif "grande école" in tl or "grandes écoles" in tl:
                    results["Type"] = "Grande École"
                elif "université" in tl or "university" in tl and "lycée" not in tl:
                    results["Type"] = "University / Higher Education"
                elif "lycée" in tl and "international" in tl:
                    results["Type"] = "International Lycée"
                elif "lycée" in tl:
                    results["Type"] = "Lycée"
                elif "conservatoire" in tl:
                    results["Type"] = "Performing Arts Institution"
                elif "british curriculum" in tl or "igcse" in tl:
                    results["Type"] = "British Curriculum School"
                elif "polo" in tl or "equestrian" in tl or "équitation" in tl:
                    results["Type"] = "Equestrian & Polo Club"
                elif "opéra" in tl or "opera" in tl:
                    results["Type"] = "Performing Arts Organisation"
                elif "tennis" in tl and ("academy" in tl or "académie" in tl or "club" in tl):
                    results["Type"] = "Tennis Academy"
                elif "yoga" in tl and "formation" in tl:
                    results["Type"] = "Yoga Education Organisation"
                elif "tournament" in tl or "grand slam" in tl or "tournoi" in tl:
                    results["Type"] = "Grand Slam Tennis Tournament & Venue"
                elif "château" in tl or "musée" in tl:
                    results["Type"] = "Heritage & Cultural Site"
                elif "football" in tl and "academy" in tl:
                    results["Type"] = "Football Academy"
                elif "sport" in tl and "club" in tl:
                    results["Type"] = "Sports & Social Club"
                else:
                    results["Type"] = "Independent School"

            soup_home = BeautifulSoup(await page.content(), "html.parser")
            for attr in [{"name": "description"}, {"property": "og:description"}]:
                tag = soup_home.find("meta", attr)
                if tag and tag.get("content"):
                    content = tag["content"][:200]
                    if not is_junk(content):
                        results["Tagline"] = content
                        break
            if not results["Tagline"]:
                candidate = extract(r"([A-Z][^.]{30,150}\.)")
                if candidate and not is_junk(candidate):
                    results["Tagline"] = candidate

            scraped_quote = scrape_quote_from_soup(soup_home)
            if scraped_quote and not is_junk(scraped_quote):
                results["Quote"] = scraped_quote

            image_set = set()
            await harvest_images(page, url, image_set)
            results["Images"] = [img for img in image_set if img][:10]

        except Exception as e:
            print(f"Home page error: {e}")
            await browser.close()
            return None

        # ── SUBPAGE CRAWL ─────────────────────────────────────────────────────
        soup  = BeautifulSoup(await page.content(), "html.parser")
        links = soup.find_all("a", href=True)
        queue     = []
        seen_urls = {url.rstrip("/")}

        targets = {
            "about": [
                "about", "history", "welcome", "mission", "who-we-are",
                "our-story", "overview", "vision", "values", "ethos",
                "histoire", "présentation", "presentation", "qui-sommes-nous",
                "notre-histoire", "valeurs", "notre-école", "notre-etablissement",
                "a-propos", "apropos", "mot-du-directeur", "le-lycée",
                "l-ecole", "le-college", "notre-lycee",
            ],
            "outcomes": [
                "results", "destination", "university", "achievements",
                "résultats", "resultats", "orientations", "grandes-ecoles",
                "parcoursup", "bac", "baccalauréat", "taux-de-réussite",
                "palmarès", "palmares", "exam", "college",
            ],
            "fees": [
                "fees", "tuition", "financial", "cost", "charges",
                "scolarite", "scolarité", "frais", "tarifs", "tarif",
                "inscription", "coût", "cout",
            ],
            "admission": [
                "admissions", "apply", "entry", "register", "enroll",
                "how-to-apply", "application", "join-us", "registration",
                "admission", "candidature", "rejoindre", "postuler",
                "portes-ouvertes", "open-day",
            ],
            "facilities": [
                "facilities", "campus", "sport", "arts", "library",
                "infrastructure", "resources", "activities", "co-curricular",
                "infrastructures", "équipements", "equipements", "installations",
                "vie-scolaire", "activités", "activites", "parascolaire",
                "programs", "academics", "programmes", "pédagogie",
            ],
        }

        for link in links:
            href, text = link["href"], link.get_text().lower().strip()
            full_url   = urljoin(url, href).split("#")[0].rstrip("/")
            if full_url not in seen_urls and url.split("/")[2] in full_url:
                if any(k in text or k in href.lower() for cat in targets.values() for k in cat):
                    queue.append((full_url, text))
                    seen_urls.add(full_url)

        used_sentences: set = set()

        def _register(text: str):
            if not text:
                return
            for s in re.split(r'(?<=[.!?])\s+', text.strip()):
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and len(key) > 20:
                    used_sentences.add(key)

        def dedup_text(text: str) -> str:
            if not text:
                return text
            parts = re.split(r'(?<=[.!?])\s+', text.strip())
            fresh = []
            for s in parts:
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and key not in used_sentences and not is_junk(s):
                    fresh.append(s.strip())
                    used_sentences.add(key)
            return " ".join(fresh)

        def assign_field(field_name: str, text: str, max_sent: int = 2):
            if results.get(field_name) and not is_junk(results[field_name]):
                return
            cleaned = dedup_text(clean_to_sentences(text, max_sent))
            if cleaned and not is_junk(cleaned):
                results[field_name] = cleaned

        for _field in ["About", "Tagline", "Philosophy", "Outcomes", "Admissions", "Quote"]:
            _register(results.get(_field, ""))

        used_paras_global: set = set()

        # Strict per-metric keyword lists
        PERF_KWS = {
            "Coaching Credentials":   [
                "professeur", "teacher", "faculty", "certified", "credential",
                "coach", "enseignant", "expertise", "qualif", "diplôm",
                "expérience", "licensed", "formation des", "staff qualif",
            ],
            "Student Wellbeing":      [
                "wellbeing", "pastoral", "santé", "accompagnement",
                "vie scolaire", "counselor", "welfare", "suivi élève",
                "soutien psychologique", "orientation", "bien-être",
            ],
            "Academic Integration":   [
                "curriculum", "igcse", "a-level", "ib", "bac", "baccalauréat",
                "programme pédagogique", "academic", "subject", "matière",
                "enseignement", "formation", "cours de", "syllabus",
            ],
            "Competitive Pathway":    [
                "university", "grande école", "concours", "résultats",
                "parcoursup", "prépa", "achievement", "ranking",
                "taux de réussite", "mention", "classement", "université",
            ],
            "Facilities & Resources": [
                "facilities", "campus", "bibliothèque", "laboratoire",
                "library", "lab", "pool", "gymnasium", "court",
                "salle", "terrain", "stade", "infrastructure", "bâtiment",
            ],
            "Ongoing Accountability": [
                "progress", "suivi", "évaluation", "bilan", "tracking",
                "monitoring", "feedback", "report", "conseil de classe",
                "bulletin", "assessment", "progress report",
            ],
        }

        # Junk skip for performance fields
        PERF_SKIP = [
            "restauration", "repas sont préparés", "cantine", "cuisin",
            "archives administratives", "vitrines d'exposition",
            "don de l'intégralité", "mésentente au sein",
            "cette expression fut utilisée", "jules ferry",
            "tarif plein", "tarif réduit", "moins de 5 ans",
            "ouverture du parc", "par téléphone",
            "les années 1970", "mai 1968",
            "injured to his wrist", "forced to withdraw",
            "le jardin des chefs", "michelin",
            "1,200 m² terrace", "entry lists for all",
            "article will be regularly updated",
            "régional d", "s'incline", "victoire de l'entente",
            "bnp paribas", "renault", "engie",
            "nos conseillers sont disponibles",
            "in.tune brings together", "norwegian academy",
            "building an ecologically", "hec has set up the first green zone",
            "sustainability & organizations",
            "work package 1", "coordinated by the",
            "créez votre compte", "identifiez-vous",
            "accept all.*button", "reject all.*button",
            "after joining the french resistance",
            "toute inscription est conditionnée",
            "ent | portail cdi",
        ]

        for target_url, link_text in queue[:15]:
            try:
                await page.goto(target_url, wait_until="domcontentloaded", timeout=30000)
                await asyncio.sleep(2)
                main          = page.locator("main").first
                content_scope = main if await main.is_visible() else page
                paragraphs    = await content_scope.locator("p").all_text_contents()
                clean_paras   = [
                    t.strip() for t in paragraphs
                    if len(t.strip()) >= 60
                    and not any(x in t.lower() for x in ["cookie", "privacy", "menu", "navigation", "footer", "identifiez-vous"])
                    and not is_junk(t)
                ]
                global_text_pool.extend(clean_paras)
                full_text = " ".join(clean_paras)
                page_text = await page.inner_text("body")
                sub_soup  = BeautifulSoup(await page.content(), "html.parser")

                await harvest_images(page, url, image_set)
                results["Images"] = [img for img in image_set if img][:10]

                if any(k in link_text or k in target_url for k in targets["about"]):
                    clean_about = [p for p in clean_paras
                                   if not is_junk(p)
                                   and not p.lower().startswith((
                                       "please fill", "please note", "click here",
                                       "merci d'indiquer", "nos conseillers",
                                       "toute inscription", "créez votre compte",
                                   ))]
                    assign_field("About", " ".join(clean_about), 2)
                    teach_kws  = ["curriculum", "teaching", "learning", "approach",
                                  "education", "ethos", "method", "ib", "philosophy",
                                  "pédagogie", "pedagogie", "enseignement", "méthode",
                                  "programme", "philosophie", "formation"]
                    teach_para = [p for p in clean_paras
                                  if any(k in p.lower() for k in teach_kws)
                                  and not is_junk(p)]
                    assign_field("Philosophy", " ".join(teach_para), 2)
                    if not results["Quote"]:
                        q = scrape_quote_from_soup(sub_soup)
                        if q and not is_junk(q):
                            results["Quote"] = q
                    if not results.get("Founded"):
                        sub_cands = extract_founded_from_text(page_text)
                        if sub_cands:
                            sub_cands.sort(key=lambda x: (-x[0], x[1]))
                            results["Founded"] = str(sub_cands[0][1])

                if not results.get("Students"):
                    for pat in student_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            num_str = re.sub(r'[\s,]', '', m.group(1))
                            try:
                                num = int(num_str)
                                if 50 <= num <= 50000 and not (2020 <= num <= 2030):
                                    results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                                    break
                            except ValueError:
                                pass

                if not results.get("Ratio"):
                    for pat in ratio_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            results["Ratio"] = m.group(1).replace(" ", "")
                            break

                if any(k in link_text or k in target_url for k in targets["outcomes"]):
                    outcome_kws = ["université", "university", "grandes écoles", "bac",
                                   "accepted", "acceptance", "placement", "graduate",
                                   "alumni", "destination", "enrolled", "résultats"]
                    outcome_paras = [p for p in clean_paras
                                     if any(k in p.lower() for k in outcome_kws)
                                     and not is_junk(p)]
                    assign_field("Outcomes", " ".join(outcome_paras) or full_text, 2)
                    if not results["Quote"]:
                        q = scrape_quote_from_soup(sub_soup)
                        if q and not is_junk(q):
                            results["Quote"] = q

                if any(k in link_text or k in target_url for k in targets["fees"]):
                    if not results.get("Fees"):
                        for pat in fee_patterns:
                            m = re.search(pat, page_text, re.I)
                            if m:
                                fee_str = m.group(0).strip()
                                nums = re.findall(r'\d+', fee_str.replace(' ','').replace('\xa0',''))
                                if nums and 100 <= int(nums[0]) <= 100000 and not (2000 <= int(nums[0]) <= 2030):
                                    results["Fees"] = fee_str
                                    break

                if any(k in link_text or k in target_url for k in targets["admission"]):
                    adm_kws = ["apply", "application", "admission", "register",
                               "candidature", "inscription", "rejoindre", "postuler",
                               "open house", "portes ouvertes", "shadow day"]
                    filtered = [p for p in clean_paras
                                if any(k in p.lower() for k in adm_kws)
                                and not is_junk(p)]
                    assign_field("Admissions", " ".join(filtered), 2)
                    dl = re.search(
                        r'(?:deadline|closes?|date limite|clôture)\s*[:\-]?\s*([^\n.]{10,60})',
                        page_text, re.I
                    )
                    if dl and results["AppDeadline"].startswith("Rolling"):
                        val = dl.group(1).strip()[:80]
                        has_month = re.search(
                            r'\b(january|february|march|april|may|june|july|august|'
                            r'september|october|november|december|jan|feb|mar|'
                            r'janvier|février|mars|avril|mai|juin|juillet|août|'
                            r'septembre|octobre|novembre|décembre)\b', val, re.I)
                        has_date = re.search(r'\b\d{1,2}[\/\-]\d{1,2}', val)
                        if (has_month or has_date) and not is_junk(val):
                            results["AppDeadline"] = val
                    if not results.get("Ages"):
                        for pat in age_patterns:
                            m = re.search(pat, page_text, re.I)
                            if m:
                                try:
                                    g = m.groups()
                                    if len(g) >= 2 and g[0] and g[1]:
                                        lo, hi = int(g[0]), int(g[1])
                                        if lo < hi and lo >= 2 and hi <= 25:
                                            results["Ages"] = f"{lo}–{hi}"
                                except Exception:
                                    pass
                                if results["Ages"]:
                                    break

                if any(k in link_text or k in target_url for k in targets["facilities"]):
                    for metric, kws in PERF_KWS.items():
                        if not results["Performance"].get(metric):
                            matched = [
                                p for p in clean_paras
                                if any(k in p.lower() for k in kws)
                                and p not in used_paras_global
                                and not is_junk(p)
                                and not any(sk in p.lower() for sk in PERF_SKIP)
                                and len(p.split()) >= 15
                            ]
                            if matched:
                                chosen = matched[:1]
                                val    = dedup_text(clean_to_sentences(" ".join(chosen), 2))
                                if val and len(val.split()) >= 12 and not is_junk(val):
                                    results["Performance"][metric] = val
                                    used_paras_global.update(chosen)
                    if not results["Quote"]:
                        q = scrape_quote_from_soup(sub_soup)
                        if q and not is_junk(q):
                            results["Quote"] = q

            except Exception as e:
                print(f"  Skipping {target_url}: {e}")
                continue

        results["Images"] = list(image_set)[:10]
        for field, fallback in FALLBACKS.items():
            if not results.get(field):
                results[field] = fallback
        if not results["City"]:
            results["City"] = "Paris, France"

        if not results["Quote"]:
            print(f"  No quote found — generating AI quote…")
            results["Quote"] = generate_ai_quote(
                results["Name"], results["Type"], results["City"], results["About"]
            )

        await browser.close()

        # ── STEP 1: Fill main fields ──────────────────────────────────────────
        print(f"\n  Running OpenAI main field gap-fill…")
        results = openai_fill_missing_fields(results)

        # ── STEP 2: Force-fill ALL 6 performance fields ───────────────────────
        print(f"\n  Running AI force-fill for performance fields…")
        results = openai_force_fill_performance(results)

        # ── POST SANITIZATION ─────────────────────────────────────────────────
        for field in ["Name", "Tagline", "Founded", "City", "Ages", "Students",
                      "Ratio", "Fees", "Type", "About", "Philosophy", "Outcomes",
                      "Admissions", "Quote"]:
            if results.get(field):
                results[field] = re.sub(r'\s+', ' ', str(results[field])).strip()
        for field in ["About", "Philosophy", "Outcomes", "Admissions"]:
            if results.get(field) and len(results[field].split()) > 60:
                results[field] = clean_to_sentences(results[field], 2)
        PERF_METRICS = ["Coaching Credentials", "Student Wellbeing", "Academic Integration",
                        "Competitive Pathway", "Facilities & Resources", "Ongoing Accountability"]
        for metric in PERF_METRICS:
            val = results["Performance"].get(metric, "")
            if val and len(val.split()) > 50:
                results["Performance"][metric] = clean_to_sentences(val, 2)

        # ── PRINT SUMMARY ─────────────────────────────────────────────────────
        SEP        = "—" * 55
        city_short = results["City"].split(",")[0]
        print(f"\n{SEP}")
        print(f"Elite › {city_short} › {results['Name']}")
        print(SEP)
        print(f"\n  Type       : {results['Type']}")
        print(f"  Ages       : {results['Ages'] or 'N/A'}")
        print(f"  Students   : {results['Students'] or 'N/A'}")
        print(f"  Ratio      : {results['Ratio'] or 'N/A'}")
        print(f"  Founded    : {results['Founded'] or 'N/A'}")
        print(f"  City       : {results['City']}")
        print(f"  Annual Fee : {results['Fees']}")
        print(f"  Images     : {len(results['Images'])} found")
        for i, img in enumerate(results["Images"], 1):
            print(f"    {i}. {img}")
        print(f"\n  About      : {results['About']}")
        print(f"\n  Quote      : \"{results['Quote']}\"")
        print(f"\n  Philosophy : {results['Philosophy']}")
        print(f"\n  Outcomes   : {results['Outcomes']}")
        print(f"\n  Admissions : {results['Admissions']}")
        print(f"\nPerformance Metrics:")
        for k, v in results["Performance"].items():
            kw = PERF_SHORT.get(k, k)
            print(f"\n  [{kw}] {k}:")
            print(f"  {v or '❌ STILL MISSING'}")
        print(f"\n  Website : {results['URL']}")
        print(SEP)

        # ── SAVE ──────────────────────────────────────────────────────────────
        wp_code     = generate_wp_blocks(results)
        school_slug = re.sub(r"[^a-z0-9]+", "_", results["Name"].lower())[:40].strip("_")
        out_wp      = f"{school_slug}_wp_blocks.txt"
        out_json    = f"{school_slug}_data.json"

        with open(out_wp, "w", encoding="utf-8") as f:
            f.write(wp_code)

        save_data               = {k: v for k, v in results.items() if k != "Images"}
        save_data["ImageCount"] = len(results.get("Images", []))
        save_data["ImageURLs"]  = results.get("Images", [])
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(save_data, f, ensure_ascii=False, indent=2)

        print(f"\n  WP blocks  → {out_wp}")
        print(f"  JSON data  → {out_json}")
        return results


# ── CITY FALLBACK (body-text only, SF/non-Paris cities blocked) ──────────────

PARIS_BODY_HINTS = {
    "saint-germain-en-laye":  "Saint-Germain-en-Laye, France",
    "saint germain en laye":  "Saint-Germain-en-Laye, France",
    "neuilly-sur-seine":      "Neuilly-sur-Seine, France",
    "neuilly sur seine":      "Neuilly-sur-Seine, France",
    "chantilly":              "Chantilly, France",
    "versailles":             "Versailles, France",
    "fontainebleau":          "Fontainebleau, France",
    "boulogne-billancourt":   "Boulogne-Billancourt, France",
    "jouy-en-josas":          "Jouy-en-Josas, France",
    "jouy en josas":          "Jouy-en-Josas, France",
    "levallois":              "Levallois-Perret, France",
    "paris":                  "Paris, France",
    "île-de-france":          "Paris, France",
}

def _detect_city_from_body(body_text: str, url: str) -> str:
    url_lower = url.lower()
    for hint, city in PARIS_BODY_HINTS.items():
        if hint in url_lower:
            return city
    sample = body_text[:5000].lower()
    for hint in sorted(PARIS_BODY_HINTS, key=len, reverse=True):
        if hint in sample:
            return PARIS_BODY_HINTS[hint]
    return "Paris, France"


# ── URLS — deduplicated, canonical versions only ───────────────────────────────

school_urls = [
    "https://alsacienne.org",
    "https://janson-de-sailly.fr",
    "https://lycee-international-stgermain.com/",
    "https://breteuil.fr",
    "https://stanislas.fr",
    "https://www.psg.fr/en/psg-academy",
    "https://www.rolandgarros.com/en-us/",       # one URL only, no duplicate
    "https://racingclubdefrance.fr/fr/tennis",
    "https://www.poloclubchantilly.com/home",
    "https://axiom-academic.com",
    "https://cours-thales.fr",
    "https://icp.fr",
    "https://sciencespo.fr",
    "https://www.hec.edu/en/hec-academy/youth-school-programs",
    "https://operadeparis.fr",
    "https://www.conservatoiredeparis.fr/en",
    "https://rye-yoga.fr/",
    "https://ecolejeanninemanuel.org/en/home/",
    "https://www.saintemariedeneuilly.fr/en/",
]


async def run_multiple_schools():
    all_results = []
    for url in school_urls:
        print(f"\n{'=' * 60}")
        print(f"  Scraping: {url}")
        print(f"{'=' * 60}")
        try:
            result = await extract_school_data(url)
            if result:
                all_results.append(result)
            await asyncio.sleep(3)
        except Exception as e:
            print(f"  Failed: {url} → {e}")

    master_file = "all_schools_data_paris.json"
    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n{'=' * 60}")
    print(f"All done! {len(all_results)} institutions scraped.")
    print(f"Master JSON → {master_file}")
    print(f"{'=' * 60}")
    return all_results


if __name__ == "__main__":
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    try:
        loop = asyncio.get_running_loop()
        import nest_asyncio
        nest_asyncio.apply()
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            future   = pool.submit(asyncio.run, run_multiple_schools())
            all_data = future.result()
    except RuntimeError:
        asyncio.run(run_multiple_schools())


  Scraping: https://alsacienne.org
  ✓ Known institution: École Alsacienne | City locked: Paris, France
Connecting to: https://alsacienne.org...
  ⚠ Page body too short — scrolling to trigger JS render…
  No quote found — generating AI quote…
  ✓ AI quote generated for École Alsacienne

  Running OpenAI main field gap-fill…
  → OpenAI filling 7 main field(s): Ratio, Fees, About, Philosophy, Outcomes, Admissions, Tagline
  ✓ OpenAI filled: Ratio, About, Philosophy, Outcomes, Admissions, Tagline

  Running AI force-fill for performance fields…
  → AI force-filling 6 performance field(s): Coaching Credentials, Student Wellbeing, Academic Integration, Competitive Pathway, Facilities & Resources, Ongoing Accountability
  ✓ AI filled performance: Coaching Credentials, Student Wellbeing, Academic Integration, Competitive Pathway, Facilities & Resources, Ongoing Accountability

———————————————————————————————————————————————————————
Elite › Paris › École Alsacienne
———————————————————————————